# Deep Research Agent — Demo Notebook

This notebook demonstrates the LangChain Deep Research Agent backed by PostgreSQL.

## Prerequisites

1. **PostgreSQL running** — start with `docker compose up -d`
2. **Environment variables** — copy `.env.example` → `.env` and fill in your keys
3. **Dependencies installed** — run `uv sync`


## 1. Setup

In [ ]:
import sys, os

# Make sure the project root is on the path when running from the notebooks/ directory
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

from dotenv import load_dotenv
load_dotenv('../.env')

from config import DATABASE_URL, OPENAI_MODEL
print(f'Model : {OPENAI_MODEL}')
print(f'DB    : {DATABASE_URL}')

## 2. Verify PostgreSQL Connection

In [ ]:
import psycopg

try:
    with psycopg.connect(DATABASE_URL) as conn:
        row = conn.execute('SELECT version()').fetchone()
        print('Connected:', row[0])
except Exception as e:
    print('Connection failed — is docker compose up?', e)

## 3. Create the Agent

In [ ]:
from langchain_deepagent import DeepResearchAgent
from langchain_deepagent.backends import PostgresBackend
from langchain_openai import ChatOpenAI
from config import DATABASE_URL, OPENAI_API_KEY, OPENAI_MODEL

llm = ChatOpenAI(model=OPENAI_MODEL, api_key=OPENAI_API_KEY)

# PostgreSQL backend: agent state + virtual filesystem all persisted in the DB
backend = PostgresBackend(connection_string=DATABASE_URL)

agent = DeepResearchAgent(llm=llm, backend=backend)
print('Agent ready:', agent)

## 4. Run a Research Query

In [ ]:
query = "What are the latest advances in quantum computing as of 2025?"

result = await agent.ainvoke(query)
print(result)

## 5. Resume a Previous Run (Checkpoint Demo)

Because the backend is PostgreSQL, any interrupted run can be resumed by re-creating the agent with the same backend — the state is persisted automatically.

In [ ]:
# Re-create agent (simulates a restart)
agent2 = DeepResearchAgent(llm=llm, backend=backend)

# The virtual filesystem in PostgreSQL retains all previously written research artifacts
result2 = await agent2.ainvoke("Summarize what you already know about quantum computing.")
print(result2)